In [6]:
import sympy as sp
from IPython.display import display, Math

# Configuración visual
sp.init_printing(use_latex='mathjax')
t, w = sp.symbols('t w', real=True)
j_math = sp.I
j_sym = sp.Symbol('j')
u_sym = sp.Function('u')

def u(argumento):
    return sp.Heaviside(argumento)

def format_math(expr):
    # Truco para que se dibuje la 'u' y la 'j' bonitas en pantalla
    return sp.latex(expr.replace(sp.Heaviside, u_sym).subs(sp.I, j_sym))

def generador_corrimiento_tiempo(f_t_examen):
    print("==================================================")
    print(" ⏱️ DECONSTRUCTOR DE TEOREMA: CORRIMIENTO EN EL TIEMPO")
    print("==================================================\n")

    display(Math(rf"\text{{1. Función original del examen: }} f(t) = {format_math(f_t_examen)}"))

    # ALGORITMO DE DETECCIÓN: Buscamos cuánto se movió la función (t0)
    t0 = 0
    for arg in sp.preorder_traversal(f_t_examen):
        if isinstance(arg, sp.Heaviside):
            # Resolvemos lo que está adentro de la 'u' para encontrar el corrimiento
            sol = sp.solve(arg.args[0], t)
            if sol:
                t0 = sol[0]
            break

    if t0 == 0:
        print("\n--> No se detectó ningún corrimiento (t0 = 0). ¡Usa la transformada directa!")
        return

    print(f"\nPASO 1: Identificación del Desplazamiento")
    print(f"Al observar la función, identificamos la forma estándar f(t - t0).")
    print(f"Despejando el argumento, obtenemos que el corrimiento temporal es t0 = {t0}.")

    # Extrayendo la base (sustituimos t por t + t0 para limpiar la función)
    f_base = sp.simplify(f_t_examen.subs(t, t + t0))
    print(f"\nPASO 2: Extracción de la Función Base")
    print("Reconstruimos la función original sin el corrimiento:")
    display(Math(rf"f_{{base}}(t) = {format_math(f_base)}"))

    print(f"\nPASO 3: Transformada de la Función Base")
    print("Por definición o tablas, la transformada de esta base causal es:")
    try:
        # Calculamos la base usando integración sin condiciones
        res_int = sp.integrate(f_base * sp.exp(-j_math*w*t), (t, -sp.oo, sp.oo), conds='none')
        if isinstance(res_int, sp.Piecewise):
            res_int = res_int.args[0][0]
        
        # Truco de racionalización para que la 'jw' quede abajo positiva
        num, den = sp.fraction(sp.simplify(res_int))
        num_format = sp.simplify(num * j_math)
        den_format = sp.simplify(den * j_math)
        
        display(Math(rf"F_{{base}}(\omega) = \frac{{{format_math(num_format)}}}{{{format_math(den_format)}}}"))
    except Exception as e:
        print(f"Error al calcular la base: {e}")
        return

    print(f"\nPASO 4: Aplicación del Teorema")
    print("El Teorema de Corrimiento en el Tiempo dicta que un desplazamiento implica multiplicar por una exponencial en frecuencia:")
    display(Math(rf"\mathfrak{{F}}\{{f(t - t_0)\}} = F_{{base}}(\omega) \cdot e^{{-j\omega t_0}}"))
    
    # Resultado Final
    F_final_num = num_format * sp.exp(-j_math * w * t0)
    F_final_den = den_format
    
    print("\n==================================================")
    print(" RESULTADO FINAL F(w) (Copia todo esto en tu hoja):")
    display(Math(rf"F(\omega) = \frac{{{format_math(F_final_num)}}}{{{format_math(F_final_den)}}}"))
    print("==================================================")

# ==========================================
# 📝 CONFIGURACIÓN DEL EXAMEN
# ==========================================
# Mete tu problema con 'u' aquí:
mi_f_tiempo = u(t+4) * sp.exp(-(t+4))

generador_corrimiento_tiempo(mi_f_tiempo)

 ⏱️ DECONSTRUCTOR DE TEOREMA: CORRIMIENTO EN EL TIEMPO



<IPython.core.display.Math object>


PASO 1: Identificación del Desplazamiento
Al observar la función, identificamos la forma estándar f(t - t0).
Despejando el argumento, obtenemos que el corrimiento temporal es t0 = -4.

PASO 2: Extracción de la Función Base
Reconstruimos la función original sin el corrimiento:


<IPython.core.display.Math object>


PASO 3: Transformada de la Función Base
Por definición o tablas, la transformada de esta base causal es:


<IPython.core.display.Math object>


PASO 4: Aplicación del Teorema
El Teorema de Corrimiento en el Tiempo dicta que un desplazamiento implica multiplicar por una exponencial en frecuencia:


<IPython.core.display.Math object>


 RESULTADO FINAL F(w) (Copia todo esto en tu hoja):


<IPython.core.display.Math object>

# GUÍA MAESTRA MA2N: COMANDO CENTRAL DE FOURIER

Esta guía es tu árbol de decisiones para el examen. Define exactamente cuál de tus 5 Jupyter Notebooks usar dependiendo de lo que pida el Ing. Saquimuz, cómo configurarlo, y cómo evitar las trampas clásicas.

---

## INVENTARIO DE TU ARSENAL (Tus 5 Scripts)

1. **`definiciondetallada.ipynb` (El Músculo):** Integra a la fuerza bruta usando la Identidad de Euler. Ideal para funciones por tramos, pulsos y casos donde los teoremas fallan.
2. **`asistenteTeoremas.ipynb` (La Caja Negra):** Usa Laplace internamente para darte respuestas instantáneas (Directas e Inversas) 100% simplificadas. 
3. **`PasosInversa.ipynb` (Deconstructor Inverso):** Genera el procedimiento de texto a copiar en el examen cuando te dan una $F(\omega)$ y debes regresar a $f(t)$ usando el Teorema de Corrimiento en Frecuencia.
4. **`EscalonUnitario.ipynb` (Deconstructor de Tiempo):** Genera el procedimiento de texto para Transformadas Directas que usan el Teorema de Corrimiento en el Tiempo (funciones con $u$ desplazadas).
5. **`AplicadorTeoremas.ipynb` (Redactor Manual):** Tu herramienta teórica de respaldo si necesitas aplicar corrimientos metiendo el valor de $w_0$ a mano.

---

## ÁRBOL DE DECISIONES: ¿Qué script uso en el examen?

Lee el enunciado del problema y sigue esta ruta:

### CASO A: Me piden la Transformada INVERSA (Me dan $F(\omega)$ y piden $f(t)$)
* **Flujo a seguir:**
  1. Abre `asistenteTeoremas.ipynb` (Modo Inverso). Mete la función y obtén la respuesta final (ej. `exp(-3*t)*cos(10*t)`).
  2. Abre `PasosInversa.ipynb`. Mete la pregunta en `F_examen` y la respuesta del paso anterior en `f_caja_negra`.
  3. Ejecuta y copia el procedimiento de fracciones parciales y corrimientos a tu hoja.

### CASO B: Me piden Transformada DIRECTA con un Teorema de Tiempo (Me dan $u$ desplazada)
* *Ejemplo:* Encuentre la transformada de $u(t+4)e^{-(t+4)}$ usando teoremas.
* **Flujo a seguir:**
  1. Abre `EscalonUnitario.ipynb`.
  2. Escribe la función tal como está: `mi_f_tiempo = u(t+4) * sp.exp(-(t+4))`.
  3. Ejecuta y copia el texto generado a tu hoja.

### CASO C: Me piden Transformada DIRECTA por Definición (O la función es "mutante")
* *Ejemplo:* Pulsos unitarios ($-2 \le t \le 2$), o funciones donde el desplazamiento no coincide (ej. $u(t-2)e^{-t}$).
* **Flujo a seguir:**
  1. Abre `definiciondetallada.ipynb`.
  2. Traduce el problema a los límites de `mis_tramos` (Ver sección "Manejo del Escalón" abajo).
  3. Ajusta el interruptor `forzar_cosenos` y ejecuta. Copia las integrales a tu hoja.

---

## TRUCOS DE SUPERVIVENCIA PARA CONFIGURAR LOS SCRIPTS

### 1. El Interruptor de `definiciondetallada.ipynb`
Saber poner `True` o `False` en `forzar_cosenos` evita que el script arroje resultados extraños.
* **Pon `True` SI:** La función original tiene senos/cosenos, **O** si los límites de integración son simétricos (ej. de $-2$ a $2$).
* **Pon `False` SI:** Es una función exponencial normal ($e^{-at}$), polinomios ($1-t^2$), o va al infinito.

### 2. El Manejo del Escalón $u(t)$ en `definiciondetallada.ipynb`
Si el problema tiene una $u(t)$ "mutante" (ej. $u(t-2)e^{-t}$) no uses el de teoremas, usa este script y traduce la $u$ cambiando los límites numéricos. La $u(t)$ **NO** se escribe en el código, solo modifica el intervalo:
* **$u(t)$ normal:** Significa que va de $0$ a infinito.
  `mis_tramos = [(sp.exp(-t), 0, sp.oo)]`
* **$u(t - 2)$ (Atraso):** Significa que va de $2$ a infinito.
  `mis_tramos = [(sp.exp(-t), 2, sp.oo)]`
* **$u(t + 4)$ (Adelanto):** Significa que va de $-4$ a infinito.
  `mis_tramos = [(sp.exp(-t), -4, sp.oo)]`

### 3. La Regla de Oro del Signo $w_0$
Si por alguna razón necesitas usar tu `AplicadorTeoremas.ipynb` manual, ten cuidado con el signo.
* **Plantilla teórica:** $\dots \cdot e^{+j \omega_0 t}$
* Si tu examen dice: Multiplique por $e^{-j \alpha t}$
* Tienes que igualar los exponentes: $+j \omega_0 t = -j \alpha t$
* **Conclusión:** Lo que debes meter en la variable del código es `w0 = -alpha`. (Si lo metes positivo, el script te arruinará el signo de la respuesta).

---

## RESUMEN DE TU METODOLOGÍA PARA EL 100
1. **La Caja Negra primero:** Si el problema no te exige un método particular, mete la función a `asistenteTeoremas.ipynb` para saber a qué respuesta debes llegar.
2. **Confía en la definición:** Si en medio del examen un teorema te confunde o los corrimientos no encajan, mete la función a `definiciondetallada.ipynb`. Demostrar el cálculo integral manualmente garantiza tus puntos de procedimiento con el ingeniero.
